### Running the soft validation on /exp_fig_1_debug_run_1

In [ ]:
import sys
import json
import importlib
import csv
from pathlib import Path
import time

# Ensure project root is importable (since notebook is in analysis/)
ROOT_DIR = Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

# Reload soft_validator to ensure latest version
import utils.soft_validator
importlib.reload(utils.soft_validator)
from utils.soft_validator import soft_validate, SoftValidationResult

print("=" * 100)
print("SOFT VALIDATOR ANALYSIS")
print("=" * 100)
print()

# Path to experiment results (relative to project root)
#results_dir = ROOT_DIR / "results/exp_fig_1_debug_run_1_after_soft_validator"
results_dir = ROOT_DIR / "results/exp_fig_3_updated_run_1"
json_files = sorted(results_dir.glob("*.json"))

# CSV output file - save in analysis/ directory
csv_output = Path(f"soft_validator_results_{results_dir.name}_o3.csv")

# Create CSV and write header if file doesn't exist
if not csv_output.exists():
    with open(csv_output, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['log_file', 'task_class', 'deterministic_result', 'soft_validator_flag', 'justification', 'last_tool_error_type', 'last_tool_error_explanation'])

print(f"Processing {len(json_files)} execution logs...\n")
print(f"Writing results to: {csv_output}")
print("-" * 100)

# Track imported tasks to avoid re-importing
imported_tasks = {}

for json_file in json_files:
    # Load execution log
    with open(json_file, 'r') as f:
        execution_log = json.load(f)
    
    # Extract task info
    task_module = execution_log.get('task_module')
    task_class = execution_log.get('task_class')
    deterministic_result = execution_log.get('task_result', {}).get('task_success', None)
    
    # Create task key
    task_key = f"{task_module}.{task_class}"
    
    # Import task class if not already imported
    if task_key not in imported_tasks:
        module_path = f"tasks.fhir_tasks_modular.{task_module}_modular"
        mod = importlib.import_module(module_path)
        task_cls = getattr(mod, task_class if task_class.endswith("Modular") else f"{task_class}Modular")
        imported_tasks[task_key] = task_cls
    else:
        task_cls = imported_tasks[task_key]
    
    # Run soft validation
    soft_result = soft_validate(task_cls, execution_log)
    
    # Write to CSV
    with open(csv_output, 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([
            json_file.name,
            task_class,
            deterministic_result,
            soft_result.flag,
            soft_result.justification,
            soft_result.last_tool_error_type,
            soft_result.last_tool_error_explanation
        ])
    
    # Display results
    print(f"\nFile: {json_file.name}")
    print(f"Task: {task_class}")
    print(f"Deterministic Result: {deterministic_result}")
    print(f"Soft Validator Result: {soft_result.flag}")
    print(f"Justification: {soft_result.justification}")
    print(f"Last Tool Error Type: {soft_result.last_tool_error_type}")
    print(f"Last Tool Error Explanation: {soft_result.last_tool_error_explanation}")
    print("-" * 100)


print(f"\n✓ Results saved to: {csv_output}")


In [ ]:
import pandas as pd
from pathlib import Path

# Read the CSV file (results_dir relative to project root)
results_dir = ROOT_DIR / "results/exp_fig_1_debug_run_1"
csv_file = Path(f"soft_validator_results_{results_dir.name}.csv")

df = pd.read_csv(csv_file)

print("=" * 100)
print("SOFT VALIDATOR FLAG COUNTS")
print("=" * 100)
print()

# Count each flag
flag_counts = df['soft_validator_flag'].value_counts().sort_index()

print(flag_counts)
print()
print(f"Total: {len(df)} evaluations")
